In [1]:
import pandas as pd
import numpy as np
import ast
import re
from collections import Counter, defaultdict
from itertools import combinations

# Load your datasets
articles = pd.read_csv("article_company_index.csv")
companies = pd.read_csv("nyse_nasdaq_companies_with_revenue_clenaed_JP.csv")

In [2]:
print(articles.columns)
print(companies.columns)

articles[["article_id", "company"]].head()

Index(['article_id', 'date', 'title', 'url', 'company'], dtype='str')
Index(['company', 'company_id', 'exchange', 'ticker', 'industry', 'market_cap',
       'market_cap_date', 'company_url', 'revenue', 'revenue_date',
       'search_name'],
      dtype='str')


,article_id,company
0,0,"['BBC', 'Honda', 'Renault', 'Red Bull', 'the R..."
1,1,"['GLOBE NEWSWIRE', 'ADDvantage Technologies Gr..."
2,2,"['FTSE', 'Helios Investment Partners', 'the Lo..."
3,3,['BRIEF-Pareteum Awarded']
4,4,"['TSX', '/PRNewswire/ - Jaguar Mining Inc', 'J..."


In [20]:
LEGAL_SUFFIXES = r"""
inc|inc\.|corporation|corp|corp\.|company|co|co\.|group|plc|
ltd|ltd\.|limited|holdings|holding|sa|s\.a\.|ag|nv|n\.v\.|
se|spa|s\.p\.a\.|asa|ab|gmbh|lp|llp|llc|l\.l\.c\.|bv|b\.v\.
"""

suffix_re = re.compile(rf"\b({LEGAL_SUFFIXES})\b", re.IGNORECASE | re.VERBOSE)

def normalize_name(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    if not x:
        return None

    # Apple Inc’s -> Apple Inc
    x = x.replace("’s", "").replace("'s", "")
    x = x.replace("’", "'")

    # Reuters ticker style: AAPL.O -> AAPL
    x = re.sub(r"\.[A-Z]+$", "", x)

    x = x.lower()
    x = x.replace("&", " and ")

    # remove legal suffixes
    x = suffix_re.sub(" ", x)

    # remove punctuation
    x = re.sub(r"[^a-z0-9 ]+", " ", x)

    # collapse spaces
    x = re.sub(r"\s+", " ", x).strip()

    return x or None

In [21]:
alias_rows = []

for _, row in companies.iterrows():
    company_id = row["company_id"]
    company_name = row["company"]

    for col in ["company", "search_name", "ticker"]:
        if col not in companies.columns:
            continue

        value = row[col]

        if pd.isna(value):
            continue

        alias = normalize_name(value)

        if alias is None:
            continue

        # Avoid dangerous tiny aliases like F, T, A, ON
        if len(alias) < 3:
            continue

        alias_rows.append({
            "alias": alias,
            "company_id": company_id,
            "company": company_name,
            "source": col
        })

alias_df = pd.DataFrame(alias_rows).drop_duplicates()

alias_df.head()

,alias,company_id,company,source
0,honda,Q9584,Honda,company
1,honda,Q9584,Honda,search_name
2,hmc,Q9584,Honda,ticker
3,nissan,Q20165,Nissan,company
4,nissan,Q20165,Nissan,search_name


In [22]:
alias_counts = alias_df.groupby("alias")["company_id"].nunique()

ambiguous_aliases = set(alias_counts[alias_counts > 1].index)

alias_df_clean = alias_df[~alias_df["alias"].isin(ambiguous_aliases)].copy()

alias_to_company = dict(zip(alias_df_clean["alias"], alias_df_clean["company_id"]))

print("Aliases:", len(alias_df))
print("Ambiguous aliases removed:", len(ambiguous_aliases))
print("Clean aliases:", len(alias_to_company))

Aliases: 1708
Ambiguous aliases removed: 3
Clean aliases: 1101


In [23]:
def parse_company_list(x):
    if pd.isna(x):
        return []

    try:
        value = ast.literal_eval(x)
        if isinstance(value, list):
            return value
        return []
    except Exception:
        return []

articles["company_list"] = articles["company"].apply(parse_company_list)

articles[["article_id", "company_list"]].head()

,article_id,company_list
0,0,"[BBC, Honda, Renault, Red Bull, the Red Bull-o..."
1,1,"[GLOBE NEWSWIRE, ADDvantage Technologies Group..."
2,2,"[FTSE, Helios Investment Partners, the London ..."
3,3,[BRIEF-Pareteum Awarded]
4,4,"[TSX, /PRNewswire/ - Jaguar Mining Inc, JAG, C..."


In [24]:
def match_companies_in_article(company_names):
    matched = set()

    for name in company_names:
        alias = normalize_name(name)

        if alias in alias_to_company:
            matched.add(alias_to_company[alias])

    return matched

articles["matched_company_ids"] = articles["company_list"].apply(match_companies_in_article)

articles[["article_id", "company_list", "matched_company_ids"]].head()

,article_id,company_list,matched_company_ids
0,0,"[BBC, Honda, Renault, Red Bull, the Red Bull-o...",{Q9584}
1,1,"[GLOBE NEWSWIRE, ADDvantage Technologies Group...",{}
2,2,"[FTSE, Helios Investment Partners, the London ...","{Q154950, Q219508}"
3,3,[BRIEF-Pareteum Awarded],{}
4,4,"[TSX, /PRNewswire/ - Jaguar Mining Inc, JAG, C...",{}


In [25]:
articles["n_matched_companies"] = articles["matched_company_ids"].apply(len)

print("Articles with at least 1 matched company:")
print((articles["n_matched_companies"] >= 1).sum())

print("Articles with at least 2 matched companies:")
print((articles["n_matched_companies"] >= 2).sum())

print("Unique matched companies:")
print(len(set().union(*articles["matched_company_ids"])))

Articles with at least 1 matched company:
11485
Articles with at least 2 matched companies:
3500
Unique matched companies:
514


In [26]:
company_counts = Counter()

for company_ids in articles["matched_company_ids"]:
    company_counts.update(company_ids)

company_lookup = companies.set_index("company_id")["company"].to_dict()

company_count_df = pd.DataFrame([
    {
        "company_id": company_id,
        "company": company_lookup.get(company_id, company_id),
        "article_count": count
    }
    for company_id, count in company_counts.items()
])

company_count_df = company_count_df.sort_values("article_count", ascending=False)

company_count_df.head(50)

,company_id,company,article_count
10,Q1472929,"Nasdaq, Inc.",1524
18,Q312,Apple Inc.,684
23,Q3884,Amazon,587
8,Q66,Boeing,434
33,Q193326,Goldman Sachs,409
44,Q334204,Morgan Stanley,394
19,Q2283,Microsoft,371
108,Q66048,Deutsche Bank,343
12,Q478214,Tesla,274
49,Q20800404,Alphabet Inc.,245


In [ ]:
edge_counts = Counter()

for company_ids in articles["matched_company_ids"]:
    company_ids = sorted(company_ids)

    if len(company_ids) < 2:
        continue

    for a, b in combinations(company_ids, 2):
        edge_counts[(a, b)] += 1

edge_df = pd.DataFrame([
    {
        "source": a,
        "target": b,
        "weight": weight,
        "source_name": company_lookup.get(a, a),
        "target_name": company_lookup.get(b, b)
    }
    for (a, b), weight in edge_counts.items()
])

edge_df = edge_df.sort_values("weight", ascending=False)

edge_df.head(20)

,source,target,weight,source_name,target_name
69,Q312,Q3884,119,Apple Inc.,Amazon
15,Q2283,Q312,88,Microsoft,Apple Inc.
268,Q544847,Q790060,82,Qualcomm,Broadcom
84,Q60238941,Q7414,75,Fox Corporation,The Walt Disney Company
83,Q1113804,Q60238941,72,Comcast,Fox Corporation
46,Q193326,Q334204,69,Goldman Sachs,Morgan Stanley
126,Q2283,Q3884,66,Microsoft,Amazon
388,Q219508,Q334204,63,Citigroup,Morgan Stanley
19,Q3884,Q483551,61,Amazon,Walmart
65,Q1113804,Q7414,58,Comcast,The Walt Disney Company


In [ ]:
alias_df_clean.to_csv("company_aliases_auto.csv", index=False)
company_count_df.to_csv("company_article_counts.csv", index=False)
edge_df.to_csv("company_comention_edges_all.csv", index=False)